# Unsafe-Net detector retrain (Kaggle GPU)

Two arms:
- **Baseline** (sound): retrain the 6 movable entities at imgsz 960 on GPU. Primary goal = recover **Forklift_Safe_Load** (170 labels, 0 detected by the CPU-trained model) + lift per-class mAP.
- **Zones** (experimental): add detectable zone boxes (classes 2,3) from the static polygons, then retrain. Static per-camera zones may not transfer to a held-out camera (the LCO leak L2 showed) — the downstream leave-camera-out eval is the judge.

**Before running:** create a Kaggle Dataset named `unsafenet-detector` from the folder produced by `prepare-upload.sh`, and add it as an input (Add Input -> your dataset). Enable **GPU T4** (Settings -> Accelerator). Turn Internet on only if you did not bundle `yolo11s.pt`.


In [ ]:
# 0. Environment + auto-discover the dataset input (robust to slug, nesting depth, AND un-extracted .zip)
!pip -q install 'ultralytics==8.4.27'
import torch, os, glob, zipfile
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('inputs mounted:', sorted(os.path.basename(r) for r in glob.glob('/kaggle/input/*')) or 'NONE')

# The payload marker is dataset/data.yaml. Find it at ANY depth -> IN = the dir that holds dataset/.
def _find_marker(root):
    h = glob.glob(f'{root}/**/dataset/data.yaml', recursive=True)
    return h[0] if h else None

marker = _find_marker('/kaggle/input')

# self-heal: Kaggle auto-extracts a *Dataset* .zip, but a .zip added as a *file* stays zipped ->
# no dataset/data.yaml ever appears. Detect that lone .zip and extract it into the writable working dir.
if not marker:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        dest = '/kaggle/working/unsafenet-detector'
        print(f'\nno extracted dataset found; auto-extracting {zips[0]}\n  -> {dest} ...')
        os.makedirs(dest, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(dest)
        marker = _find_marker(dest)

if not marker:
    print('\n--- /kaggle/input tree (2 levels) ---')
    for r in sorted(glob.glob('/kaggle/input/*')):
        print(os.path.basename(r) + '/')
        for c in sorted(glob.glob(f'{r}/*'))[:40]:
            print('   ', os.path.basename(c) + ('/' if os.path.isdir(c) else ''))
    raise AssertionError(
        'dataset/data.yaml not found. FIX: right panel -> Add Input -> search & add your '
        '"unsafenet-detector" Dataset. If it is not listed, create it first: Datasets -> New Dataset '
        '-> upload kaggle-detector-retrain/upload/unsafenet-detector.zip (Kaggle auto-extracts it). '
        'See the /kaggle/input tree above for what is currently mounted.')

IN = os.path.dirname(os.path.dirname(marker))
SC = f'{IN}/scripts'
W8 = f'{IN}/yolo11s.pt' if os.path.exists(f'{IN}/yolo11s.pt') else 'yolo11s.pt'
print('\nusing input:', IN)
print('contents   :', sorted(os.listdir(IN)))
assert os.path.isdir(SC), f'scripts/ missing under {IN} - reupload the full zip (it bundles scripts/).'

In [ ]:
# 0b. Build a corrected baseline root (fixes the baked-in `path:`; NO dataset re-upload needed).
#     Makes /kaggle/working/base/dataset a complete, self-contained root: images/labels symlinked to the
#     read-only mount, plus a data.yaml whose `path:` points at itself. Works with any bundled script version.
import os, glob
# recover IN if cell 0 was not run in this session (avoids a confusing NameError here)
try:
    IN
except NameError:
    _h = (glob.glob('/kaggle/input/**/dataset/data.yaml', recursive=True)
          or glob.glob('/kaggle/working/**/dataset/data.yaml', recursive=True))
    assert _h, 'dataset not found - run cell 0 (env + auto-discover) first.'
    IN = os.path.dirname(os.path.dirname(_h[0]))
    print('recovered IN =', IN)
BASE = '/kaggle/working/base'
bds = f'{BASE}/dataset'
os.makedirs(bds, exist_ok=True)
for sub in ('images', 'labels'):
    link = f'{bds}/{sub}'
    if not os.path.lexists(link):
        os.symlink(os.path.abspath(f'{IN}/dataset/{sub}'), link)
NAMES = ['Person_Authorized', 'Person_Unauthorized', 'Safe_Walkway', 'Restricted_Zone',
         'Panel_Closed', 'Panel_Opened', 'Forklift_Safe_Load', 'Forklift_Overload']
with open(f'{bds}/data.yaml', 'w') as f:
    f.write(f"path: {os.path.abspath(bds)}\ntrain: images/train\nval: images/val\nnc: {len(NAMES)}\nnames:\n"
            + ''.join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
assert os.path.isdir(f'{bds}/images/val'), f'val images not reachable at {bds}/images/val'
print('baseline root ready:', bds, '->', sorted(os.listdir(bds)))

In [ ]:
# 1. (optional) smoke test — 1 epoch, tiny res, ~2 min; confirms the pipeline before a full run.
#     Uses {BASE} (the corrected root built in cell 0b), NOT {IN}: the mount's baked-in data.yaml `path:`
#     is wrong for this slug, so pointing at the raw mount fails even the smoke run. Run cell 0b first.
!python {SC}/train-detector-gpu.py --smoke --data_root {BASE} --model {W8} --project /kaggle/working/runs_smoke

In [ ]:
# 2. Baseline full train (2-stage: 20 frozen + 40 finetune). ~3-5h on a T4 at 960/b16.
#    Lower --epochs2 or --imgsz to fit the 9h Kaggle GPU session cap; drop --batch to 8 on OOM.
!python {SC}/train-detector-gpu.py \
    --data_root {BASE} --model {W8} \
    --imgsz 960 --batch 16 --epochs1 20 --epochs2 40 \
    --project /kaggle/working/runs_baseline

## Arm B — zones (experimental)

In [ ]:
# 3. Generate zone-box labels (classes 2,3) into a writable copy. Prints coverage (~1350 imgs).
!python {SC}/build-zone-labels.py \
    --src_root {IN}/dataset --zones_dir {IN}/zones \
    --view_map {IN}/clip_view_map.json --dst_root /kaggle/working/dataset_zones

In [ ]:
# 4. Zone-arm train. Validates whether the detector can emit zone boxes at all (in-distribution val).
!python {SC}/train-detector-gpu.py --add_zones \
    --zones_yaml /kaggle/working/dataset_zones/data.yaml --model {W8} \
    --imgsz 960 --batch 16 --epochs1 20 --epochs2 40 \
    --project /kaggle/working/runs_zones

## Arm C — BEHAVIOUR detector (Gate 1 / task A1)

**Different detector, different label set.** Arms A/B above train the 8 *entity* classes that feed the
P3 relation-Transformer. This arm trains the 8 *behaviour* classes of the paper's Table 3 scheme
(WalkwayViol, UnauthorInt, OpenedPanel, ...) — the vocabulary Gate 1's "mAP50 >= 0.90, all 8 classes
emitted" actually refers to. Do not mix them up.

Prediction for this run is pre-registered in
`plans/2026-09-06-thesis-2week-final/gate1-detector-retrain-prediction.md`. Read it before looking at the
output, and fill in its Outcome table afterwards.

Three caveats that apply to whatever mAP comes out:
1. **OpenedPanel is degenerate** — a single constant box localises 100% of its val instances at IoU 0.5,
   so ~0.125 of mAP50 is free mass.
2. The val split shares **all 8 camera views** with train — this is in-domain mAP, not generalisation.
3. 12 of 343 val frames are byte-identical to a train frame, 60 of 62 duplicate groups have contradictory
   labels.


In [ ]:
# C1. Build the behaviour labels (calls the repo's own unsafenet/behaviour_dataset.py — no second copy)
!python {SC}/build-behaviour-labels.py --data_root {IN} --work /kaggle/working --variant tight
!python {SC}/build-behaviour-labels.py --data_root {IN} --work /kaggle/working --variant ctx
BEH = '/kaggle/working/annotation_guide'


In [ ]:
# C2. (optional) smoke test — 1 epoch, ~2 min, confirms plumbing before spending GPU quota
!python {SC}/train-behaviour-detector-gpu.py --smoke \
    --data {BEH}/dataset_behaviour_tight/data.yaml --project /kaggle/working/runs_beh_smoke


In [ ]:
# C3. Behaviour detector, `tight` arm. yolo11s/imgsz800/200ep ~1.5h on a T4. --batch 8 on OOM.
#     Swap --model yolo11m.pt for more tail-class capacity (~2x epoch time), or yolo26s.pt as a stretch run.
!python {SC}/train-behaviour-detector-gpu.py \
    --data {BEH}/dataset_behaviour_tight/data.yaml \
    --model {W8} --imgsz 800 --batch 16 --epochs 200 \
    --project /kaggle/working/runs_behaviour


In [ ]:
# C4. Behaviour detector, `ctx` arm (person boxes widened to include the disambiguating context).
!python {SC}/train-behaviour-detector-gpu.py \
    --data {BEH}/dataset_behaviour_ctx/data.yaml \
    --model {W8} --imgsz 800 --batch 16 --epochs 200 \
    --project /kaggle/working/runs_behaviour


In [ ]:
# 5. Collect weights for download (right panel -> /kaggle/working -> download best.pt)
import glob, shutil, os
os.makedirs('/kaggle/working/weights', exist_ok=True)
for tag in ('baseline', 'zones'):
    src = f'/kaggle/working/runs_{tag}/stage2_finetune/weights/best.pt'
    if os.path.exists(src):
        shutil.copy2(src, f'/kaggle/working/weights/detector_{tag}_best.pt')
        print('saved', tag, '->', f'/kaggle/working/weights/detector_{tag}_best.pt')
print('handoff: download these to the repo, re-extract relation tokens on the 516 behaviour clips,')
print('then re-run run_lco_eval.py / run_kfold_relation.py to judge each detector under LCO.')
# behaviour-detector arms + their gate JSONs
for v in ('tight', 'ctx'):
    src = f'/kaggle/working/runs_behaviour/beh_{v}_gpu/weights/best.pt'
    if os.path.exists(src):
        shutil.copy2(src, f'/kaggle/working/weights/behaviour_{v}_best.pt')
        print('saved', f'behaviour_{v}_best.pt')
    rj = f'/kaggle/working/runs_behaviour/beh_{v}_gpu_result.json'
    if os.path.exists(rj):
        shutil.copy2(rj, f'/kaggle/working/weights/beh_{v}_result.json')
        print(open(rj).read())
